# Approximate quantum Fourier transform

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0. [Locked source notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/qft/approximate_qft.ipynb).

In [ ]:
from IPython.display import SVG, display
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from examples.algorithms.field_addition_catalog import field_signature_svg

## Phase cutoff

Prepare a $b$-wire phase-gradient state with amplitudes proportional to $e^{-2\pi i k/2^b}$. Coherent addition into it supplies the required data phases. Keeping a shorter gradient drops smaller phases and reduces the modeled arithmetic cost. The transform needs no arbitrary rotations after preparing the gradient. The default precision is $\lceil\log_2 n\rceil$. An error tolerance $\epsilon$ chooses $\lceil\log_2(\pi\sqrt{2}n/\epsilon)\rceil$. Final swaps are optional. [Phase-gradient derivation](https://algassert.com/post/1620) and [approximation bound](https://arxiv.org/abs/quant-ph/0008056).

In [ ]:
from examples.algorithms.approximate_qft_catalog import approximate_qft_examples, approximate_qft_graph
from flagquantum.algorithms.primitives.approximate_qft_resource_model import (
    source_approximate_qft_signature, source_approximate_qft_formula,
    source_approximate_qft_epsilon_precision,
)

### Example instances

In [ ]:
small, epsilon_example = approximate_qft_examples()
small

In [ ]:
epsilon = 1e-5
assert epsilon_example == (50, source_approximate_qft_epsilon_precision(50, epsilon))
epsilon_example

#### Graphical signature

The unsigned data and fixed-point gradient are both through ports.

In [ ]:
display(SVG(field_signature_svg(tuple(
    (f"Approximate QFT ({n}, {p})", source_approximate_qft_signature(n, p), {})
    for n, p in (small, epsilon_example)
))))

### Call graph

The declared shift-one controlled additions group identical retained window sizes.

In [ ]:
graph = approximate_qft_graph(*small)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
print("Declared leaves:", graph.leaf_counts)

## Approximate QFT cost analysis

The locked notebook assigns four modeled T gates to each declared Toffoli. The resulting expression uses a symbolic data width and error threshold.

In [ ]:
from sympy import symbols
n, eps = symbols("n epsilon")
precision = source_approximate_qft_epsilon_precision(n, eps)
formula = source_approximate_qft_formula(n, precision)
print("Concrete source model:", source_approximate_qft_formula(*epsilon_example))
print("Modeled symbolic T:", 4 * formula.toffoli)

### Execute through append

A two-wire gradient on three data wires omits the smallest controlled phase. This sample validates the retained phases and cleanup separately from the source cost estimate.

In [ ]:
from collections import Counter
from math import pi
import torch
from flagquantum.algorithms.primitives.approximate_qft import approximate_qft_ir, approximate_qft_workspace_size
from flagquantum.circuit import Circuit
from flagquantum.compiler.registers.phase.phase_gradient import phase_gradient_state_ir
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic
from examples.algorithms.binary_field_arithmetic import FieldOperation

width, precision = 3, 2
extra = approximate_qft_workspace_size(width, precision)
gradient = tuple(range(width, width + precision))
work = tuple(range(width + precision, width + precision + extra))
body = approximate_qft_ir(tuple(range(width)), gradient, work, tuple(range(extra)))
print("Emitted transform:", Counter(op.name for op in body))
gates = (*phase_gradient_state_ir(gradient, exponent=-1), *body,
         *phase_gradient_state_ir(gradient, exponent=-1, inverse=True))
n = width + precision + extra
values = torch.arange(1, 2**width + 1, dtype=torch.float64)
state = torch.complex(values, 1-values)
state /= state.norm()
initial = torch.zeros(2**n, dtype=torch.complex128)
stride = 2**(precision + extra)
initial[::stride] = state
circuit = DynamicCircuit(n, inputs=initial, dtype=torch.complex128)
circuit.append(FieldOperation(n, gates))
result = run_dynamic(circuit, shots=4, seed=21, strategy="batched")
reference = Circuit(width, inputs=state, dtype=torch.complex128)
reference.h(0).gate("cphase", (1, 0), theta=pi/2).h(1)
reference.gate("cphase", (2, 1), theta=pi/2).h(2).swap(0, 2)
expected = torch.zeros_like(initial)
expected[::stride] = reference.state().flatten()
torch.testing.assert_close(result.final_states.reshape(4, -1), expected.expand(4, -1))